# 🖼 Selenium: скачиваем картинки и файлы

**Методы сбора и обработки данных · Selenium, занятие 3**

Selenium умеет находить элементы, но **скачивать файлы он не умеет**. Поэтому схема всегда одна:

1. Selenium находит элемент (`img`, ссылку);
2. достаём из него **адрес** файла (`src`, `href`, `data-src`);
3. скачиваем по адресу — обычно библиотекой `requests`.

Сегодня:
* 📤 картинка и скриншот всей страницы с тестового сайта — **задание LXP**;
* 🖼 миниатюра или оригинал, ленивая загрузка, прокрутка — как в Яндекс.Картинках;
* 📂 72 картинки по папкам-классам — заготовка к **КТ** (там из таких папок учат нейросеть).

### Как работать
1. Запустите **⚙️ Подготовку** (Shift+Enter), около минуты.
2. В заданиях замените `...` своим кодом и запустите ячейку.
3. Скачанные файлы видны слева, в папке 📁. Показать картинку из файла: `show_image("cat.jpg")`, а страницу в браузере — `show(driver)`.

In [ ]:
#@title ⚙️ Подготовка: запустите эту ячейку первой (около минуты) { display-mode: "form" }
import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("/usr/bin/google-chrome"):
    print("Ставим Google Chrome…")
    subprocess.run("wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb"
                   " && apt-get -qq update && apt-get -qq install -y /tmp/chrome.deb",
                   shell=True, check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "selenium"], check=True)
if IN_COLAB or not os.path.exists("common.py"):
    subprocess.run(["wget", "-q", "-O", "common.py", "https://raw.githubusercontent.com/Scr1ptG/mso_data/main/selenium-02/practice/common.py"], check=True)

import importlib, common
importlib.reload(common)
from common import start, polygon, check, show, try_selector, show_image, full_page_screenshot
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import requests, time, io
from urllib.parse import urljoin
from PIL import Image

try:
    driver.quit()   # если ячейку запускают второй раз — закрываем старый браузер
except Exception:
    pass
driver = start()
wait = WebDriverWait(driver, 10)
TEST = "http://ithub.role.ru/download_media.html"
driver.get(TEST)
print("✅ Браузер запущен. Открыта страница:", driver.title)
show(driver)

> 🆘 **Если что-то сломалось** (`invalid session id`, Colab перезапустился) — запустите **⚙️ Подготовку** ещё раз.

> ⚠️ **Если тестовая страница открывается пустой** на своём компьютере: новый Chrome сам меняет `http://` на `https://`, а у этого сайта https нет. В Подготовке это уже исправлено. В своём коде добавьте в настройки браузера:
> ```python
> options.add_argument("--disable-features=HttpsUpgrades,HttpsFirstBalancedModeAutoEnable")
> ```

---
## 1. Тестовая страница LXP

Откройте её в соседней вкладке: http://ithub.role.ru/download_media.html — и посмотрите через F12, где лежат адреса:

```html
<img src="img/cat-web.jpg">                              ← картинка: адрес в src
<a href="data/important-document.pdf">Очень важный домент</a>   ← файл: адрес в href
<a href="news.html">Страничка с новостью</a>              ← страница
```
`get_attribute("src")` и `get_attribute("href")` сразу отдают **полный** адрес: `http://ithub.role.ru/img/cat-web.jpg`.

### ✏️ Задание 1. Адрес картинки

Найдите на тестовой странице тег `img` и сохраните в `src` адрес картинки.

In [ ]:
driver.get(TEST)

img = ...  # ВАШ КОД: driver.find_element(...)
src = ...  # ВАШ КОД: img.get_attribute(...)

print(src)
check("i01", src)

<details><summary>💡 Подсказка</summary>

Картинка на странице одна, поэтому её можно найти просто по тегу: `By.TAG_NAME, "img"`.
</details>

## 2. Скачиваем: requests

```python
response = requests.get(src)          # скачали
print(response.status_code)           # 200 — всё хорошо, 403/404 — не дали или нет такого
print(response.headers["Content-Type"])  # image/jpeg

with open("cat.jpg", "wb") as f:      # "wb" — записать байты (write binary)
    f.write(response.content)         # .content — байты файла, .text — только для текста!
```

### ✏️ Задание 2. Скачать картинку (задание LXP, часть 1)

Скачайте картинку по адресу `src` из задания 1 в файл `cat.jpg`.

In [ ]:
# ВАШ КОД: response = requests.get(...), напечатать статус, записать response.content в файл cat.jpg

show_image("cat.jpg")
check("i02", "cat.jpg")

<details><summary>💡 Подсказка</summary>

Режим `"wb"`, а не `"w"`: картинка — это байты, а не текст. И пишем `response.content`, не `response.text`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
response = requests.get(src)
print("Статус:", response.status_code)
with open("cat.jpg", "wb") as f:
    f.write(response.content)
```
</details>

## 3. Скриншоты

| Что | Код | Что получится |
|---|---|---|
| Видимая часть окна | `driver.save_screenshot("visible.png")` | только то, что поместилось в окно |
| Один элемент | `img.screenshot("cat_screen.png")` | картинка **как её видно на странице** |
| Вся страница | растянуть окно на всю высоту → `save_screenshot` | вся страница целиком |

⚠️ Скриншот элемента — **не оригинал**: браузер мог уменьшить картинку, у неё может быть рамка, а сохраняется она в PNG. Оригинал скачивают по `src`.

### ✏️ Задание 3. Скриншот элемента

Сделайте скриншот одного элемента `img` в файл `cat_screen.png`. Сравните с `cat.jpg`: размер, вес файла, рамка.

In [ ]:
driver.get(TEST)
img = driver.find_element(By.TAG_NAME, "img")

# ВАШ КОД: скриншот элемента img в cat_screen.png

show_image("cat_screen.png")
show_image("cat.jpg")
check("i03", "cat_screen.png")

<details><summary>💡 Подсказка</summary>

У элемента есть метод `.screenshot("имя.png")`.
</details>

### ✏️ Задание 4. Скриншот ВСЕЙ страницы (задание LXP, часть 2)

`driver.save_screenshot()` снимает только то, что видно в окне. Страница выше окна — значит, растянем окно:
```python
height = driver.execute_script("return document.documentElement.scrollHeight")   # высота всей страницы
driver.set_window_size(1280, height + 200)   # +200: размер окна считается вместе с рамкой браузера
driver.save_screenshot("page.png")
driver.set_window_size(1280, 900)   # вернуть как было
```
Сначала посмотрите, что даёт обычный скриншот, потом сделайте полный в `page.png`.

In [ ]:
driver.get(TEST)
driver.save_screenshot("visible.png")
show_image("visible.png", width=300)   # обычный скриншот: страница обрезана снизу

# ВАШ КОД: полный скриншот в page.png

show_image("page.png", width=300)
check("i04", "page.png")

<details><summary>💡 Подсказка</summary>

Скопируйте 4 строчки из описания. Если проверка пишет «только видимая часть» — не забыли `+ 200`? В других заданиях можно просто вызвать готовую функцию `full_page_screenshot(driver, "page.png")` — внутри то же самое.
</details>

### 📤 Что сдать в LXP
Задание «Загрузите изображение с тестового сайта»: файлы **`cat.jpg`** и **`page.png`** (скачать: папка 📁 слева → ⋮ → Скачать) и ссылку на этот ноутбук (Файл → Сохранить копию на Диске → Настройки доступа → «Все, у кого есть ссылка»).

---
## 4. Не только картинки: документы и страницы

### ✏️ Задание 5. Скачать PDF

Найдите ссылку на «Очень важный домент» (да, на странице опечатка) и скачайте документ в `document.pdf`.
Ссылку удобно искать по **части** текста: `By.PARTIAL_LINK_TEXT, "важный"`.

In [ ]:
driver.get(TEST)

link = ...  # ВАШ КОД
url = ...   # ВАШ КОД: адрес документа — атрибут href
print(url)

# ВАШ КОД: скачать url в document.pdf

check("i05", "document.pdf")

<details><summary>💡 Подсказка</summary>

Всё как с картинкой: `requests.get(url)`, затем `response.content` в файл с режимом `"wb"`.
</details>

### ✏️ Задание 6. Сохранить страницу

Перейдите по ссылке «Страничка с новостью» и сохраните её HTML в `news.html`.
`driver.page_source` — это HTML страницы **после** того, как браузер выполнил JavaScript. `requests.get(url).text` даёт HTML **до** этого: на сайтах, которые рисуют страницу скриптом, там будет почти пусто.

In [ ]:
driver.get(TEST)
link = ...  # ВАШ КОД: ссылка «Страничка с новостью»
# ВАШ КОД: перейти по ней: driver.get(адрес из href)
print("Открыто:", driver.title)

# ВАШ КОД: записать driver.page_source в news.html (это текст: режим "w", encoding="utf-8")

check("i06", "news.html")

<details><summary>💡 Подсказка</summary>

Можно и `link.click()`, но сразу после загрузки страницы клик иногда «перехватывается» (`ElementClickInterceptedException`): элемент ещё двигается. `driver.get(href)` надёжнее.
</details>

---
## 5. Фотобанк: как устроены сайты с картинками

Откройте в соседней вкладке **Фотобанк Полигона**: https://scr1ptg.github.io/mso_data/polygon/gallery.html — и прокрутите вниз. Он устроен как Яндекс.Картинки:

```html
<a class="photo" href="photos/full/ball_01.jpg" data-class="ball">        ← ОРИГИНАЛ (512×512) — по ссылке
  <img class="photo__img"
       src="assets/placeholder.svg"                                    ← пока не видно — заглушка
       data-src="photos/thumb/ball_01.jpg">                            ← МИНИАТЮРА (128×128)
  <span class="photo__caption">Мячи №1</span>
</a>
```

* **Миниатюра или оригинал.** В карточке показана маленькая копия. Для датасета нужны оригиналы — они по ссылке.
* **Ленивая загрузка.** Пока карточку не видно, в `src` лежит заглушка, а настоящий адрес — в `data-src`. Браузер подставит его, когда карточка появится на экране.
* **Бесконечная прокрутка.** Сначала 12 фото. Остальные подгружаются, когда долистали до низа.

`get_attribute("data-src")` отдаёт адрес **как в коде**: `photos/thumb/ball_01.jpg`. Полный адрес собирает `urljoin(driver.current_url, адрес)`.

In [ ]:
driver.get(polygon("gallery.html"))
try_selector(driver, ".gallery a.photo")   # сколько карточек сейчас на странице?

### ✏️ Задание 7. Миниатюра или оригинал

Возьмите первую карточку. Скачайте её миниатюру (`data-src`) и оригинал (`href`) **в память** и узнайте размеры:
```python
size = Image.open(io.BytesIO(requests.get(url).content)).size   # (ширина, высота)
```

In [ ]:
driver.get(polygon("gallery.html"))
card = driver.find_element(By.CSS_SELECTOR, ".gallery a.photo")

thumb_url = ...  # ВАШ КОД: urljoin(driver.current_url, data-src картинки внутри card)
full_url = ...   # ВАШ КОД: href самой карточки
print(thumb_url, full_url, sep="\n")

thumb_size = ...  # ВАШ КОД
full_size = ...   # ВАШ КОД
print("миниатюра:", thumb_size, "| оригинал:", full_size)
check("i07", (thumb_size, full_size))

<details><summary>💡 Подсказка</summary>

Картинка лежит ВНУТРИ карточки: `card.find_element(By.TAG_NAME, "img")`. У неё берём `data-src`, у самой карточки — `href`.
</details>

## 6. Прокрутка до конца

Прокрутить страницу вниз — команда JavaScript:
```python
driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight)")
```
Новые карточки приходят не сразу. Универсальный способ (подходит и для Яндекса, где нет надписи «это всё»): листаем, ждём, считаем. **Число перестало расти — значит, дошли до конца.**
```python
before = 0
while True:
    driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight)")
    time.sleep(1.5)
    now = len(driver.find_elements(By.CSS_SELECTOR, ".gallery a.photo"))
    if now == before:
        break
    before = now
```

### ✏️ Задание 8. Все 72 ссылки на оригиналы

Долистайте фотобанк до конца и соберите в список `links` адреса **оригиналов** всех 72 фото.

⚠️ Ловушка: если брать `src` у картинок, у части окажется `placeholder.svg`. При быстрой прокрутке браузер не успел их «увидеть». `href` карточки есть всегда.

In [ ]:
driver.get(polygon("gallery.html"))

# ВАШ КОД: цикл прокрутки до конца (см. выше)

cards = driver.find_elements(By.CSS_SELECTOR, ".gallery a.photo")
links = []  # ВАШ КОД: href каждой карточки
print("Ссылок:", len(links))
srcs = [c.find_element(By.TAG_NAME, "img").get_attribute("src") for c in cards]
print("А вот src ещё не загрузились у", sum("placeholder" in s for s in srcs), "картинок")
check("i08", links)

<details><summary>💡 Подсказка</summary>

Сначала цикл прокрутки, потом `driver.find_elements` по карточкам и `get_attribute("href")` у каждой.
</details>

### ✏️ Задание 9. Датасет по папкам (заготовка к КТ)

Скачайте все 72 оригинала в папки по классам: `dataset/ball/ball_01.jpg`, `dataset/box/…`, `dataset/tree/…`.
Класс карточки лежит в атрибуте `data-class`, имя файла — конец ссылки: `url.split("/")[-1]`.

Именно в таком виде (папка = класс) данные нужны для нейросети в КТ. Пауза `time.sleep(0.1)` — вежливость к серверу.

In [ ]:
import os

cards = driver.find_elements(By.CSS_SELECTOR, ".gallery a.photo")   # страница уже долистана в задании 8
for card in cards:
    url = card.get_attribute("href")
    cls = ...  # ВАШ КОД: атрибут data-class
    name = url.split("/")[-1]
    # ВАШ КОД: создать папку dataset/cls (os.makedirs(..., exist_ok=True)), скачать url в dataset/cls/name
    time.sleep(0.1)

for cls in sorted(os.listdir("dataset")) if os.path.isdir("dataset") else []:
    print(cls, len(os.listdir(f"dataset/{cls}")))
check("i09", "dataset")

<details><summary>💡 Подсказка</summary>

Если в проверке «файлов 12» — запустите сначала задание 8: оно долистывает страницу до конца.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
os.makedirs(f"dataset/{cls}", exist_ok=True)
with open(f"dataset/{cls}/{name}", "wb") as f:
    f.write(requests.get(url).content)
```
</details>

Скачать всю папку одним архивом:
```python
import shutil; shutil.make_archive("dataset", "zip", "dataset")
from google.colab import files; files.download("dataset.zip")
```

---
## 7. Когда requests не пускают

Сайт видит, **кто** пришёл. Браузер присылает свой User-Agent и cookies (например, после входа на сайт), а `requests` — `python-requests/2.x` и без cookies. Некоторые сайты такому отвечают `403`. Решение — взять всё у браузера:

In [ ]:
print("Браузер:", driver.execute_script("return navigator.userAgent"))
print("requests:", requests.utils.default_user_agent())

session = requests.Session()
session.headers["User-Agent"] = driver.execute_script("return navigator.userAgent")
for c in driver.get_cookies():                        # cookies из Selenium → в requests
    session.cookies.set(c["name"], c["value"])
# дальше качаем через session.get(url), а не requests.get(url)

### ✏️ Задание 10. Бонус: скачать прямо через браузер

Ещё способ: попросить **сам браузер** скачать файл (JavaScript `fetch`) и вернуть байты в Python. Подходит, когда сайт пускает только «настоящий» браузер с его cookies.
Браузер возвращает текст, поэтому байты кодируются в base64 — обратно их переводит `base64.b64decode`.

Скачайте так оригинал первой фотографии в `via_browser.jpg`.

In [ ]:
import base64

JS = """
const done = arguments[arguments.length - 1];
fetch(arguments[0])
  .then(r => r.blob())
  .then(b => { const fr = new FileReader(); fr.onload = () => done(fr.result.split(',')[1]); fr.readAsDataURL(b); });
"""
driver.get(polygon("gallery.html"))
url = driver.find_element(By.CSS_SELECTOR, ".gallery a.photo").get_attribute("href")

b64 = driver.execute_async_script(JS, url)   # браузер скачал и вернул base64-строку
# ВАШ КОД: base64.b64decode(b64) → записать в via_browser.jpg

show_image("via_browser.jpg", width=200)
check("i10", "via_browser.jpg")

---
## Шпаргалка

| Задача | Как |
|---|---|
| адрес картинки | `img.get_attribute("src")`; ленивая — `data-src`; оригинал — `href` ссылки |
| относительный адрес → полный | `urljoin(driver.current_url, адрес)` |
| скачать файл | `requests.get(url).content` → `open(имя, "wb")` |
| скриншот элемента / окна | `el.screenshot("a.png")` / `driver.save_screenshot("a.png")` |
| скриншот всей страницы | окно на всю высоту `scrollHeight` → `save_screenshot` |
| HTML страницы | `driver.page_source` → `open(имя, "w", encoding="utf-8")` |
| прокрутить вниз | `driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight)")` |
| сайт не пускает requests | User-Agent и cookies из браузера в `requests.Session()` |

**Вежливо:** паузы между скачиваниями, не качаем одно и то же по сто раз, смотрим `robots.txt`.

In [ ]:
driver.quit()   # закрыть браузер в конце работы
print("Браузер закрыт")